# CORDIS — Plot Playground

Interactive notebook for exploring paper figures.  Loads any experiment result and lets you tweak:
- which algorithms appear
- figure size / aspect / typography
- which metric is plotted (for CDFs: `min_sinr_db`, `weighted_sum_scnr_db`, etc.)
- output formats and target directory

Use this instead of editing `scripts/plot_*.py`.  Once you settle on a recipe you like, you can either:
1. Leave it as a notebook (good for one-off exploration), or
2. Promote it to a `scripts/plot_<name>_<variant>.py` and add it to the regenerator metadata.

In [ ]:
# ── Setup ────────────────────────────────────────────────────────────
from pathlib import Path
import sys

# Make repo root importable (notebook may be run from anywhere)
REPO_ROOT = Path.cwd()
while not (REPO_ROOT / 'cordis').is_dir():
    if REPO_ROOT.parent == REPO_ROOT:
        raise RuntimeError('could not locate repo root with cordis/ subdir')
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))
print(f'repo root: {REPO_ROOT}')

import matplotlib.pyplot as plt
import numpy as np

from cordis.experiments.result import ExperimentResult
from cordis.plotting import (
    apply_paper_style, figsize, ALGORITHM_STYLE,
    plot_cdf, plot_sweep, plot_admm_convergence,
)

# Apply the paper style once for all subsequent figures.
apply_paper_style()

## 1. Load a result

Point at any experiment run.  By default, this picks the latest run of `sinr_cdf`.

In [ ]:
# Default: latest run of sinr_cdf.  Override with an explicit path if needed.
EXP_NAME   = 'sinr_cdf'
EXP_DIR    = None   # e.g. REPO_ROOT/'results'/'exp_sinr_cdf'/'20260518_134200'

if EXP_DIR is None:
    latest = REPO_ROOT / 'results' / f'exp_{EXP_NAME}' / 'latest'
    if not latest.exists():
        raise FileNotFoundError(f'no results found at {latest}; run `make {EXP_NAME}` first')
    EXP_DIR = latest.resolve()

result = ExperimentResult.load(EXP_DIR)
print(f'loaded: {EXP_DIR}')
print(f'kind:   {result.kind}')
print(f'metadata keys: {list(result.metadata)}')
print('available algorithms:')
for name in result.sim_result.names:
    print(f'  • {name}')

## 2. Default figure (all algorithms)

Reproduces what `scripts/plot_sinr_cdf.py` produces.

In [ ]:
fig, ax = plt.subplots(figsize=figsize(width='single', aspect=3.5/2.4))
plot_cdf(
    result.sim_result,
    metric='min_sinr_db',
    xlabel=r'min-SINR [dB]',
    ax=ax,
)
plt.show()

## 3. Filter algorithms

Display names must match `_DISPLAY` in `cordis/experiments/specs.py`:
`CORDIS-Split`, `CORDIS-ADMM`, `Centralized`, `MRT-Split`, `ZF-Split`, `RZF-Split`,
`LR-MMSE-Split`, `Global-MRT`, `Global-ZF`.

In [ ]:
PICK = [
    'CORDIS-Split',
    'CORDIS-ADMM',
    'Centralized',
    'Global-ZF',
    'Global-MRT',
]

fig, ax = plt.subplots(figsize=figsize(width='double', aspect=3.5/2.4))
plot_cdf(
    result.sim_result,
    metric='min_sinr_db',
    xlabel=r'min-SINR [dB]',
    ax=ax,
    only=PICK,
)
plt.show()

## 4. Change figure size / aspect ratio

`figsize(width=..., aspect=W/H)` understands these widths from `cordis.plotting.style.WIDTHS`:

| Key | Inches | Use for |
|---|---|---|
| `'single'` / `'half'` | 3.50 | one column of a two-column IEEE paper |
| `'third'`             | 2.40 | three-figure rows |
| `'double'`            | 7.16 | full width |

Aspect is **width/height** — bigger aspect → wider / shorter.

In [ ]:
# Wider, taller version for a presentation slide:
fig, ax = plt.subplots(figsize=figsize(width='double', aspect=7.16/4.0))
plot_cdf(
    result.sim_result,
    metric='min_sinr_db',
    xlabel=r'min-SINR [dB]',
    ax=ax,
    only=PICK,
)
ax.set_title('5-algorithm comparison — min-SINR CDF')
plt.show()

## 5. Customise per-algorithm style

`ALGORITHM_STYLE` is a dict mapping display name → matplotlib style overrides.
You can inspect it, tweak entries before plotting, or override on the axes after.

In [ ]:
# Inspect the style table
for name in PICK:
    if name in ALGORITHM_STYLE:
        print(f'{name:20s} → {ALGORITHM_STYLE[name]}')
    else:
        print(f'{name:20s} → (no entry; matplotlib defaults)')

In [ ]:
# Example: make CORDIS-ADMM thicker and dashed without touching the source file.
from copy import deepcopy
STYLE_TWEAKED = deepcopy(ALGORITHM_STYLE)
STYLE_TWEAKED['CORDIS-ADMM'].update(linewidth=2.5, linestyle='--')

# Monkey-patch for this one figure
import cordis.plotting as _cp
_saved = _cp.ALGORITHM_STYLE
_cp.ALGORITHM_STYLE = STYLE_TWEAKED

fig, ax = plt.subplots(figsize=figsize(width='single', aspect=3.5/2.4))
plot_cdf(result.sim_result, metric='min_sinr_db',
         xlabel=r'min-SINR [dB]', ax=ax, only=PICK)
plt.show()

_cp.ALGORITHM_STYLE = _saved   # restore

## 6. Plot a different metric

Each `SimResult` carries multiple metrics per algorithm.  For CDFs the common ones are:
- `min_sinr_db` — bottleneck communication SINR
- `mean_sinr_db` — average across users
- `weighted_sum_scnr_db` — sensing SCNR (if the experiment includes targets)

In [ ]:
# Discover which metrics are present
any_alg = next(iter(result.sim_result.algorithm_results.values()))
available_metrics = [m for m in ('min_sinr_db', 'mean_sinr_db',
                                 'weighted_sum_scnr_db', 'sum_scnr_db')
                     if any_alg.has_metric(m)]
print('available metrics:', available_metrics)

In [ ]:
# Compose a 2-row figure: SINR CDF on top, SCNR CDF on bottom (if present)
n_rows = len(available_metrics)
fig, axes = plt.subplots(n_rows, 1,
                         figsize=figsize(width='single',
                                         aspect=3.5/(2.2*n_rows)),
                         sharex=False)
if n_rows == 1:
    axes = [axes]

for ax, metric in zip(axes, available_metrics):
    plot_cdf(result.sim_result, metric=metric,
             xlabel=f'{metric}  [dB]', ax=ax, only=PICK)
fig.tight_layout()
plt.show()

## 7. Save the figure

Pick a name and a target directory.  The recipe below mirrors what `_plot_common.save_paper_figure` does.

In [ ]:
OUT_DIR  = REPO_ROOT / 'figures' / 'exp_sinr_cdf'
STEM     = 'sinr_cdf_5alg_playground'
FORMATS  = ['pdf', 'png']   # add 'pgf' for LaTeX inclusion

OUT_DIR.mkdir(parents=True, exist_ok=True)

fig, ax = plt.subplots(figsize=figsize(width='single', aspect=3.5/2.4))
plot_cdf(result.sim_result, metric='min_sinr_db',
         xlabel=r'min-SINR [dB]', ax=ax, only=PICK)
fig.tight_layout()

for fmt in FORMATS:
    path = OUT_DIR / f'{STEM}.{fmt}'
    fig.savefig(path, bbox_inches='tight', dpi=300 if fmt == 'png' else None)
    print(f'wrote {path.relative_to(REPO_ROOT)}')

plt.show()

## 8. Compare across runs (advanced)

Load two results (e.g. different `snr_db` values) and overlay them on the same axes.

In [ ]:
# Uncomment and adjust paths to compare two runs
#
# RUN_A = REPO_ROOT/'results'/'exp_sinr_cdf'/'20260518_120000'
# RUN_B = REPO_ROOT/'results'/'exp_sinr_cdf'/'20260518_130000'
# r_a = ExperimentResult.load(RUN_A)
# r_b = ExperimentResult.load(RUN_B)
#
# fig, ax = plt.subplots(figsize=figsize(width='single', aspect=3.5/2.4))
# plot_cdf(r_a.sim_result, metric='min_sinr_db', ax=ax, only=['CORDIS-ADMM'])
# # Re-plot with a different label by tweaking the artist
# for line in ax.lines:
#     line.set_label(line.get_label() + ' (snr_db=120)')
# plot_cdf(r_b.sim_result, metric='min_sinr_db', ax=ax, only=['CORDIS-ADMM'])
# for line in ax.lines[1:]:
#     if 'snr_db' not in line.get_label():
#         line.set_label(line.get_label() + ' (snr_db=130)')
# ax.legend()
# plt.show()